# Prompt Experiments with the Galileo Python SDK

This notebook keeps the dataset and experiment calls inline so you can see how Galileo datasets, prompt templates, and experiment runs are assembled step by step.

**Why use experiments?** Experiments help when a team needs more than a single manual prompt test. They let you run the same prompt or function over a dataset, score every row, compare prompt/model variants, and make decisions with evidence instead of intuition.

**When users benefit from experiments**

- When comparing **prompt versions** before shipping.
- When comparing **model settings** such as temperature, max tokens, or model alias.
- When checking quality against **reference answers** in a dataset.
- When evaluating output characteristics like **tone**, **BLEU/ROUGE**, or **confidence** across many examples.

**Notebook flow (aligned with the Prompt Engineering Lab scenario)**

1. **§1–3** — Bootstrap Galileo, create an evaluation dataset, and add more test cases.
2. **§4** — Create a prompt template with `{{input}}` and run a baseline prompt experiment.
3. **§5** — Run a code-based experiment with a Python function.
4. **§6** — Run an experiment with expression metrics so you can inspect style and reference-based quality.
5. **§7** — Run an experiment with different model settings plus confidence metrics, then compare results side by side in the **Experiments** UI.

Ultimately, the goal is to answer: **Which prompt or configuration performs best for my use case, and why?**


In [2]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


In [3]:
from galileo import GalileoMetrics, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.datasets import create_dataset, delete_dataset, get_dataset
from galileo.experiments import PromptRunSettings, run_experiment
from galileo.log_streams import create_log_stream
from galileo.projects import delete_project

PROJECT_NAME = 'GalileoEval_S4_Experiments'
LOG_STREAM_NAME = 'experiment-lab'
DATASET_NAME = 'prompt-lab-dataset'
BASELINE_TEMPLATE_NAME = 'concise-scientific-prompt-template'
dataset_id = None


## 1. Initialize Galileo

Bootstrap the Galileo project and an experiment lab log stream, then print console links. The rest of this notebook focuses on **Experiments**, but this step gives you a stable project home for datasets, templates, and experiment runs.


In [4]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

try:
    create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')


https://console.demo-v2.galileocloud.io/project/d8780357-ced3-4851-87ed-6a170e1b7416
https://console.demo-v2.galileocloud.io/project/d8780357-ced3-4851-87ed-6a170e1b7416/log-streams/bde2d377-b199-4e44-8d61-2f111ec76c1f


## 2. Create a Dataset


In [5]:
ds = create_dataset(
    name=DATASET_NAME,
    content=[
        {'input': 'Explain photosynthesis in one sentence.', 'output': 'Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose and oxygen.'},
        {'input': 'What causes rain?', 'output': 'Rain occurs when water vapor condenses into droplets that fall when heavy enough.'},
        {'input': 'Why is the sky blue?', 'output': 'The sky appears blue because shorter blue wavelengths scatter more than other colors.'},
        {'input': 'How do vaccines work?', 'output': 'Vaccines train the immune system to recognize and fight pathogens.'},
    ],
    project_name=PROJECT_NAME,
)
dataset_id = ds.id
str(dataset_id)


'c669ad6d-177a-44eb-8334-b13ac2d8fdd0'

## 3. Add More Dataset Rows


In [7]:
ds = get_dataset(id=str(dataset_id))
ds.add_rows([
    {'input': 'What is gravity?', 'output': 'Gravity is the force of attraction between objects with mass.'},
    {'input': 'How does WiFi work?', 'output': 'WiFi uses radio waves to transmit data wirelessly between devices and a router.'},
])
'Added 2 rows'


'Added 2 rows'

## 4. Create a Prompt Template and Run a Baseline Experiment

This is the most common experiment workflow: define a reusable prompt template with `{{input}}`, run it across the whole dataset, and score the outputs.

**What to check in the UI:** Open **Experiments** in Galileo, open this run, and review per-row scores for correctness, instruction adherence, and ground-truth adherence. This becomes your baseline for later comparisons.


In [ ]:
from galileo import Message, MessageRole
from galileo.prompts import GlobalPromptTemplates

templates = GlobalPromptTemplates()
prompt_template = templates.get(name=BASELINE_TEMPLATE_NAME)
if prompt_template is None:
    prompt_template = templates.create(
        name=BASELINE_TEMPLATE_NAME,
        template=[
            Message(role=MessageRole.system, content='Answer in exactly one sentence. Be precise and scientific.'),
            Message(role=MessageRole.user, content='{{input}}'),
        ],
        project_name=PROJECT_NAME,
    )

baseline_run = run_experiment(
    experiment_name='concise-scientific-prompt',
    dataset=DATASET_NAME,
    prompt_template=prompt_template,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.ground_truth_adherence,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o mini', temperature=0.3, max_tokens=100),
    project=PROJECT_NAME,
)
baseline_run['message']


'Concise prompt experiment complete'

## 5. Run a Code-Based Experiment

Use a code-based experiment when you want to compare a Python transformation or application function instead of a stored prompt template. This is useful for teams testing post-processing logic, formatting rules, or app-specific response generation.

**Where do experiment metrics show up?** Passing `metrics=[...]` to `run_experiment()` is the correct API. Those metrics are attached to the **experiment run** itself. In the Galileo UI they appear under **Experiments** when you open the run results, not under the log stream’s enabled-metrics settings.

**What to check in the UI:** Compare this run with the baseline prompt-template run. Does the code-based transformation improve instruction adherence, or does it reduce correctness?

In [11]:
def eli5_runner(input_text: str) -> str:
    return f"Imagine you're 5 years old. {input_text} works like a simple science trick."

function_run = run_experiment(
    experiment_name='eli5-function-experiment',
    dataset=DATASET_NAME,
    function=eli5_runner,
    metrics=[GalileoMetrics.correctness, GalileoMetrics.instruction_adherence],
    project=PROJECT_NAME,
)
function_run['message']


'Code-based experiment complete'

## 6. Run an Expression-Metrics Experiment

This step explores experiment metrics that help answer: *Does the response sound the way I want, and how close is it to the expected answer?*

- **BLEU / ROUGE** compare the generated answer to the dataset's reference `output`, so they belong in an experiment, not a log stream.
- **Input tone / Output tone** help inspect whether prompt wording or model replies have the tone you expect.

**Why a user benefits:** This is useful when a team cares not only about factual correctness, but also about presentation quality, consistency of wording, and closeness to a target answer style.

**What to check in the UI:** In **Experiments**, open this run and inspect BLEU/ROUGE together with tone metrics. Compare it against the baseline run to see whether a prompt that scores well on correctness also produces the style you want.

In [13]:
expression_run = run_experiment(
    experiment_name='expression-metrics-comparison',
    dataset=DATASET_NAME,
    prompt_template=prompt_template,
    metrics=[
        GalileoMetrics.bleu,
        GalileoMetrics.rouge,
        GalileoMetrics.input_tone,
        GalileoMetrics.output_tone,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o mini', temperature=0.3, max_tokens=100),
    project=PROJECT_NAME,
)
expression_run['message']


[]

## 7. Run a Model-Settings and Confidence Experiment

This step explores two experiment ideas together:

- **Configure model settings** with `PromptRunSettings` (for example `temperature` and `max_tokens`).
- **Check model confidence** with **Prompt perplexity** and **Uncertainty**.

**Why a user benefits:** This is useful when a team wants to compare not just prompt wording, but also *how the same prompt behaves under different model settings*. A higher temperature may produce more varied answers, but it can also increase uncertainty or reduce adherence.

**What to check in the UI:** Compare this run side by side with the baseline experiment. Look for trade-offs between correctness, instruction adherence, prompt perplexity, and uncertainty. If you run several variants, Galileo's **Experiments** view lets you compare multiple runs side by side and keep the history of prompt/model changes over time. This is the core reason to use experiments: choosing the best prompt/settings combination with evidence from a dataset, not from a single example.

In [14]:
confidence_run = run_experiment(
    experiment_name='temperature-confidence-comparison',
    dataset=DATASET_NAME,
    prompt_template=prompt_template,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.prompt_perplexity,
        GalileoMetrics.uncertainty,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o mini', temperature=0.8, max_tokens=150),
    project=PROJECT_NAME,
)
confidence_run['message']


[]

## 8. Clean Up the Dataset and Project


In [ ]:
try:
    if dataset_id is not None:
        delete_dataset(id=str(dataset_id))
    else:
        delete_dataset(name=DATASET_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

try:
    delete_project(name=PROJECT_NAME)
except Exception:
    pass
